In [1]:
%pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
def main():
    user_query = input("prompt=")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": user_query}
        ]
    )
    print(f"Bot: {response.choices[0].message.content}")

In [5]:
import requests

/Users/muhammadali/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [10]:
import requests


def get_weather(city):
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        return f"The weather in {city} is {response.text}"


In [12]:
print(get_weather("London"))

The weather in London is {
  "current_condition": [
    {
      "FeelsLikeC": "5",
      "FeelsLikeF": "41",
      "cloudcover": "70",
      "humidity": "66",
      "observation_time": "09:11 PM",
      "precipInches": "0.0",
      "precipMM": "0.0",
      "pressure": "1008",
      "pressureInches": "30",
      "temp_C": "8",
      "temp_F": "47",
      "uvIndex": "0",
      "visibility": "10",
      "visibilityMiles": "6",
      "weatherCode": "119",
      "weatherDesc": [
        {
          "value": "Cloudy "
        }
      ],
      "weatherIconUrl": [
        {
          "value": "https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png"
        }
      ],
      "winddir16Point": "W",
      "winddirDegree": "281",
      "windspeedKmph": "26",
      "windspeedMiles": "16"
    }
  ],
  "nearest_area": [
    {
      "areaName": [
        {
          "value": "Strand"
        }
      ],
      "country": [
        {
          "value": "United Kingdom

In [23]:
def get_weather(city):
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    if response.status_code == 200:
        return response.json()
response = get_weather("Koblenz")
data = response.get("current_condition")[0]
current_temp = data.get("temp_C")
feels_like = data.get("FeelsLikeC")
print(f"Weather is {current_temp} and feels like {feels_like}")

Weather is 9 and feels like 5


In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Get the current weather for a city. "
                "Use this tool whenever the user asks about "
                "current weather or temperature in a city."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city, e.g. Berlin or London"
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

In [14]:
system_prompt = """
You are a helpful AI agent with access to a weather tool.

Follow this workflow:

START:
Understand the user's request.

PLAN:
Decide whether the request requires current weather data.

ACTION:
If current weather information is required, call the get_weather tool.
Do not invent weather information.
If the request does not require weather data, answer directly.

OBSERVE:
Read the result returned by the tool.
If the tool fails, explain that the weather information could not be retrieved.

ANSWER:
Give the user a clear, natural answer based on the available information.
Briefly explain your decision when useful.
Do not reveal private chain-of-thought reasoning.

Examples:

Example 1:
User: What is the weather in Berlin?
Action: Call get_weather with city="Berlin".
Answer: Report the actual temperature and feels-like temperature.

Example 2:
User: What does feels-like temperature mean?
Action: No tool call is necessary.
Answer: Explain the concept directly.

Example 3:
User: Tell me the current temperature in Paris.
Action: Call get_weather with city="Paris".
Answer: Use the returned weather data.

Example 4:
User: Tell me a joke.
Action: No tool call is necessary.
Answer: Tell a joke without calling the weather tool.

Always use the tool when the user requests current weather data.
"""

In [15]:
import json


def main():
    user_query = input("Prompt: ")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        tools=tools,
        tool_choice="auto"
    )

    assistant_message = response.choices[0].message

    if assistant_message.tool_calls:
        tool_call = assistant_message.tool_calls[0]
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "get_weather":
            city = arguments["city"]
            weather_result = get_weather(city)
            print(f"Weather for {city}: {weather_result}")

            follow_up_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query},
                {
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [
                        {
                            "id": tool_call.id,
                            "type": "function",
                            "function": {
                                "name": function_name,
                                "arguments": json.dumps(arguments)
                            }
                        }
                    ]
                },
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(weather_result)
                }
            ]

            final_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=follow_up_messages
            )
            print(final_response.choices[0].message.content)
        else:
            print(assistant_message.content or "No response")
    else:
        print(assistant_message.content or "No response")


In [16]:
main()

It seems your message didn't come through. Could you please provide your request again?
